# ESC-50 / UrbanSound8K overlap detection

Part of the code for *Danger Detection through Audio Captioning and Classification Fusion* (Electronics, MDPI). Paper section: 3.1.

Finds UrbanSound8K clips that duplicate ESC-50 clips in the corresponding classes (dog/dog_bark, siren, car_horn) using three criteria: shared Freesound source ID, identical SHA-256 file hash, and near-duplicate waveforms (normalized cross-correlation of RMS envelopes ≥ 0.95). The union removes 240 clips.

**Environment.** Written for Google Colab (CPU is enough); see `environment/`. Paths point to a Google Drive layout. Edit the path cell to your own copy of the datasets. Random seed: 42.

**Inputs.** ESC-50 and UrbanSound8K audio and metadata.

**Outputs.** List of overlapping clips and the criterion that matched (data/dedup/).

**Order.** Mount Drive → load metadata and define the class pairs → apply the three rules → union → save the per-rule hits and the final removal list.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install librosa soundfile tqdm


In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import librosa
from tqdm.auto import tqdm

ESC50_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ESC-50-master")
URBAN_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/UrbanSound8k")

esc = pd.read_csv(ESC50_ROOT / "meta" / "esc50.csv")
urban = pd.read_csv(URBAN_ROOT / "metadata" / "UrbanSound8K.csv")

CLASS_PAIRS = {
    "dog": "dog_bark",
    "siren": "siren",
    "car_horn": "car_horn",
}

esc["src_file"] = esc["src_file"].astype(str)
urban["fsID"] = urban["fsID"].astype(str)

esc["path"] = esc["filename"].apply(lambda x: ESC50_ROOT / "audio" / x)
urban["path"] = urban.apply(
    lambda r: URBAN_ROOT / "audio" / f"fold{int(r['fold'])}" / r["slice_file_name"],
    axis=1,
)


In [ ]:
def file_sha256(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def load_audio(path, sr=16000):
    y, _ = librosa.load(path, sr=sr, mono=True)
    y = y.astype(np.float32)
    if len(y) == 0:
        return y
    y = y - np.mean(y)
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak
    return y

def fingerprint_audio(path, sr=16000, hop=512):
    y = load_audio(path, sr=sr)
    if len(y) < sr // 2:
        return None
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop)[0]
    rms = rms - np.mean(rms)
    norm = np.linalg.norm(rms)
    if norm > 0:
        rms = rms / norm
    return rms.astype(np.float32)

def max_normalized_correlation(a, b):
    if a is None or b is None:
        return np.nan
    if len(a) < 3 or len(b) < 3:
        return np.nan
    if len(a) > len(b):
        a, b = b, a
    a = a - np.mean(a)
    b = b - np.mean(b)
    a_norm = np.linalg.norm(a)
    if a_norm == 0:
        return np.nan
    best = -1.0
    for start in range(0, len(b) - len(a) + 1):
        window = b[start:start + len(a)]
        denom = a_norm * np.linalg.norm(window)
        if denom == 0:
            continue
        score = float(np.dot(a, window) / denom)
        if score > best:
            best = score
    return best


In [ ]:
shared_source_rows = []
exact_hash_rows = []
near_waveform_rows = []

for esc_class, urban_class in CLASS_PAIRS.items():
    e = esc[esc["category"] == esc_class].copy()
    u = urban[urban["class"] == urban_class].copy()

    # Rule 1: shared source ID
    shared_ids = sorted(set(e["src_file"]) & set(u["fsID"]))
    if shared_ids:
        shared_hits = u[u["fsID"].isin(shared_ids)].copy()
        shared_hits["rule"] = "shared_source_id"
        shared_hits["esc_class"] = esc_class
        shared_hits["urban_class"] = urban_class
        shared_source_rows.append(shared_hits)

    # Rule 2: exact file hash
    e_hashes = {}
    for _, er in tqdm(e.iterrows(), total=len(e), desc=f"ESC hash {esc_class}"):
        e_hashes[file_sha256(er["path"])] = er

    for _, ur in tqdm(u.iterrows(), total=len(u), desc=f"Urban hash {urban_class}"):
        h = file_sha256(ur["path"])
        if h in e_hashes:
            exact_hash_rows.append({
                "rule": "exact_hash",
                "esc_class": esc_class,
                "urban_class": urban_class,
                "esc_filename": e_hashes[h]["filename"],
                "urban_filename": ur["slice_file_name"],
                "urban_fsID": ur["fsID"],
                "hash": h,
            })

    # Rule 3: near waveform
    shared_ids_for_near = sorted(set(e["src_file"]) & set(u["fsID"]))
    e2 = e[e["src_file"].isin(shared_ids_for_near)].copy()
    u2 = u[u["fsID"].isin(shared_ids_for_near)].copy()

    esc_fp = {}
    for _, er in tqdm(e2.iterrows(), total=len(e2), desc=f"ESC fp {esc_class}"):
        esc_fp[er["filename"]] = fingerprint_audio(er["path"])

    urban_fp = {}
    for _, ur in tqdm(u2.iterrows(), total=len(u2), desc=f"Urban fp {urban_class}"):
        urban_fp[ur["slice_file_name"]] = fingerprint_audio(ur["path"])

    for _, er in tqdm(e2.iterrows(), total=len(e2), desc=f"Near compare {esc_class}"):
        candidates = u2[u2["fsID"] == er["src_file"]]
        for _, ur in candidates.iterrows():
            score = max_normalized_correlation(
                esc_fp[er["filename"]],
                urban_fp[ur["slice_file_name"]],
            )
            if pd.notna(score) and score >= 0.95:
                near_waveform_rows.append({
                    "rule": "near_waveform",
                    "esc_class": esc_class,
                    "urban_class": urban_class,
                    "esc_filename": er["filename"],
                    "urban_filename": ur["slice_file_name"],
                    "urban_fsID": ur["fsID"],
                    "similarity": score,
                })


In [ ]:
shared_source_df = (
    pd.concat(shared_source_rows, ignore_index=True)
    if shared_source_rows else
    pd.DataFrame(columns=list(urban.columns) + ["rule", "esc_class", "urban_class"])
)

exact_hash_df = pd.DataFrame(exact_hash_rows)
near_waveform_df = pd.DataFrame(near_waveform_rows)

shared_urban_files = set(shared_source_df["slice_file_name"]) if not shared_source_df.empty else set()
exact_urban_files = set(exact_hash_df["urban_filename"]) if not exact_hash_df.empty else set()
near_urban_files = set(near_waveform_df["urban_filename"]) if not near_waveform_df.empty else set()

all_remove_files = sorted(shared_urban_files | exact_urban_files | near_urban_files)

print("Shared-source removals:", len(shared_urban_files))
print("Exact-hash removals:", len(exact_urban_files))
print("Near-waveform removals:", len(near_urban_files))
print("Union to remove:", len(all_remove_files))
print(all_remove_files[:100])


In [ ]:
urban_removed = urban[urban["slice_file_name"].isin(all_remove_files)].copy()
urban_clean = urban[~urban["slice_file_name"].isin(all_remove_files)].copy()

print("Original UrbanSound8K rows:", len(urban))
print("Removed rows:", len(urban_removed))
print("Remaining rows:", len(urban_clean))

print("\nRemoved by class:")
display(urban_removed["class"].value_counts().sort_index())

display(
    urban_removed[["slice_file_name", "fold", "class", "fsID", "start", "end"]]
    .sort_values(["class", "fsID", "slice_file_name"])
)


In [ ]:
OUT = Path("/content/drive/MyDrive/Colab Notebooks/Datasets Overlap Review")
OUT.mkdir(parents=True, exist_ok=True)

shared_source_df.to_csv(OUT / "rule1_shared_source_id_hits.csv", index=False)
exact_hash_df.to_csv(OUT / "rule2_exact_hash_hits.csv", index=False)
near_waveform_df.to_csv(OUT / "rule3_near_waveform_hits.csv", index=False)
urban_removed.to_csv(OUT / "UrbanSound8K_removed_union_of_all_overlap_rules.csv", index=False)
urban_clean.to_csv(OUT / "UrbanSound8K_clean_union_of_all_overlap_rules.csv", index=False)

print("Saved all reports to:", OUT)


In [ ]:
from pathlib import Path
import pandas as pd

OUT = Path("/content/drive/MyDrive/Colab Notebooks/Datasets Overlap Review")

def read_csv_safe(path, fallback_cols=None):
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame(columns=fallback_cols or [])
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame(columns=fallback_cols or [])

shared_source_df = read_csv_safe(
    OUT / "rule1_shared_source_id_hits.csv",
    ["slice_file_name", "class", "fsID", "fold", "start", "end", "rule", "esc_class", "urban_class"]
)

exact_hash_df = read_csv_safe(
    OUT / "rule2_exact_hash_hits.csv",
    ["rule", "esc_class", "urban_class", "esc_filename", "urban_filename", "urban_fsID", "hash"]
)

near_waveform_df = read_csv_safe(
    OUT / "rule3_near_waveform_hits.csv",
    ["rule", "esc_class", "urban_class", "esc_filename", "urban_filename", "urban_fsID", "similarity"]
)

shared_files = shared_source_df[["slice_file_name", "class", "fsID", "fold", "start", "end"]].copy()
shared_files["rule_shared_source_id"] = 1

if exact_hash_df.empty:
    exact_files = pd.DataFrame(columns=["slice_file_name", "class", "fsID", "rule_exact_hash"])
else:
    exact_files = exact_hash_df[["urban_filename", "urban_class", "urban_fsID"]].copy()
    exact_files = exact_files.rename(columns={
        "urban_filename": "slice_file_name",
        "urban_class": "class",
        "urban_fsID": "fsID",
    })
    exact_files["rule_exact_hash"] = 1

if near_waveform_df.empty:
    near_files = pd.DataFrame(columns=["slice_file_name", "class", "fsID", "similarity", "rule_near_waveform"])
else:
    near_files = near_waveform_df[["urban_filename", "urban_class", "urban_fsID", "similarity"]].copy()
    near_files = near_files.rename(columns={
        "urban_filename": "slice_file_name",
        "urban_class": "class",
        "urban_fsID": "fsID",
    })
    near_files["rule_near_waveform"] = 1

final_remove = shared_files.merge(
    exact_files,
    on=["slice_file_name", "class", "fsID"],
    how="outer"
).merge(
    near_files,
    on=["slice_file_name", "class", "fsID"],
    how="outer"
)

for col in ["rule_shared_source_id", "rule_exact_hash", "rule_near_waveform"]:
    if col in final_remove.columns:
        final_remove[col] = final_remove[col].fillna(0).astype(int)

final_remove = final_remove.sort_values(["class", "fsID", "slice_file_name"]).reset_index(drop=True)

print("Total clips to remove:", len(final_remove))
print("Shared-source hits:", int(final_remove["rule_shared_source_id"].sum()) if "rule_shared_source_id" in final_remove else 0)
print("Exact-hash hits:", int(final_remove["rule_exact_hash"].sum()) if "rule_exact_hash" in final_remove else 0)
print("Near-waveform hits:", int(final_remove["rule_near_waveform"].sum()) if "rule_near_waveform" in final_remove else 0)

display(final_remove.head(100))

final_remove.to_csv(OUT / "UrbanSound8K_final_union_removal_list_with_rules.csv", index=False)
print("Saved:", OUT / "UrbanSound8K_final_union_removal_list_with_rules.csv")


In [ ]:
display(final_remove[["slice_file_name", "class", "fsID", "rule_shared_source_id", "rule_exact_hash", "rule_near_waveform"]])
